
# 10 - Feature Engineering (Event-Aware, cleaned)

Este notebook toma el dataset merged y construye una versión **más compacta, coherente y lista para ML**.

## Objetivo
Reducir redundancia, preservar señal útil y dejar un conjunto de variables interpretable para modelado.

## Decisiones principales
- Se conservan variables fisiológicas y productivas que describen directamente a la vaca.
- Los eventos se representan con **flags** y con variables tipo **days since event**, porque esto captura tanto ocurrencia como memoria temporal del evento.
- El clima se resume con **THI**, porque combina temperatura y humedad en una sola variable con sentido zootécnico.
- La alimentación se compacta con **ratios** y variables de resumen, para evitar colinealidad fuerte entre ingredientes.
- El tiempo se representa con variables **cíclicas** (`sin/cos`) para evitar discontinuidades artificiales.


## 1. Importaciones

In [1]:

from pathlib import Path
import numpy as np
import pandas as pd



## 2. Carga del dataset base

Este notebook asume que ya existe un dataset merged proveniente del notebook de integración de fuentes.

Se busca el archivo en varias rutas comunes para que el notebook sea más robusto a distintos entornos.


In [2]:

def resolve_path(filename: str) -> Path:
    candidates = [
        Path(filename),
        Path("./") / filename,
        Path("/mnt/data") / filename,
        Path("./outputs") / filename,
        Path("../outputs") / filename,
        Path("../data/processed") / filename,
        Path("../data/interim") / filename,
    ]
    for p in candidates:
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(f"No se encontró {filename}")

PATH_DATASET = resolve_path("training_dataset.parquet")
print("Dataset encontrado en:", PATH_DATASET)

df = pd.read_parquet(PATH_DATASET)
df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()

print("Shape inicial:", df.shape)
display(df.head())


Dataset encontrado en: /Users/josemanuelromoperedo/Desktop/ProyectoGranja/data/processed/training_dataset.parquet
Shape inicial: (18345, 52)


,cow_id,date,ordenos_dia,produccion_kg,duracion_total_min,di_mean,dd_mean,td_mean,ti_mean,ubre,...,THI,heat_stress_flag,dow_sin,dow_cos,doy_sin,doy_cos,carbs_ratio,protein_ratio,fiber_ratio,is_pregnant
0,1204,2022-06-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,67.799947,0,-0.781831,0.623490,0.455907,-0.890028,0.0,0.0,0.0,0.0
1,1204,2022-06-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,66.428090,0,0.974928,-0.222521,0.409356,-0.912375,0.0,0.0,0.0,0.0
2,1204,2022-07-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,66.204862,0,-0.433884,-0.900969,0.025818,-0.999667,0.0,0.0,0.0,0.0
3,1204,2022-08-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,62.226701,0,0.000000,1.000000,-0.680773,-0.732494,0.0,0.0,0.0,0.0
4,1204,2022-09-27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,60.098584,0,0.781831,0.623490,-0.996659,-0.081676,0.0,0.0,0.0,0.0



## 3. Orden temporal y copia de trabajo

Se ordena por `cow_id` y `date` porque varias transformaciones dependen del historial temporal de cada vaca:
- lags
- rolling means
- days since event
- forward fill por vaca


In [3]:

df = df.sort_values(["cow_id", "date"]).reset_index(drop=True).copy()



## 4. Utilidades

### `days_since_event_from_date`
Esta función calcula los días reales transcurridos desde el último evento.

### ¿Por qué usar fechas reales y no distancia entre filas?
Porque en un dataset temporal puede haber días faltantes.  
Contar filas no siempre equivale a contar días.


In [4]:

def days_since_event_from_date(group: pd.DataFrame, event_col: str, date_col: str = "date") -> pd.Series:
    last_date = pd.NaT
    out = []

    for _, row in group.iterrows():
        event_value = row[event_col]
        current_date = row[date_col]

        if pd.isna(event_value):
            event_value = 0

        if event_value == 1:
            last_date = current_date
            out.append(0.0)
        elif pd.isna(last_date):
            out.append(np.nan)
        else:
            out.append((current_date - last_date).days)

    return pd.Series(out, index=group.index)



## 5. Manejo de eventos binarios

Las columnas tipo `*_flag` representan eventos clínicos o reproductivos.

### Decisión
Se reemplazan sus `NaN` por `0`.

### Justificación
En este contexto, un `NaN` suele significar **ausencia de registro del evento en esa fecha**, no una medición continua faltante.


In [5]:

event_flag_cols = [c for c in df.columns if c.endswith("_flag")]

if event_flag_cols:
    df[event_flag_cols] = (
        df.groupby("cow_id")[event_flag_cols]
          .transform(lambda x: x.fillna(0))
    )

display(df[event_flag_cols].head() if event_flag_cols else pd.DataFrame())


,mastitis_flag,digestive_flag,vaccine_flag,reproductive_flag,abortion_flag,antibiotic_flag,hormone_flag,dryoff_related_flag,fresh_cow_flag,rumen_bolus_flag,heat_flag,insemination_flag,calving_flag,dryoff_flag,days_since_mastitis_flag,days_since_insemination_flag,days_since_calving_flag,days_since_abortion_flag,heat_stress_flag
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0



## 6. Variable de embarazo persistente

### Decisión
Se construye `is_pregnant` a partir de `pregnancy_positive` con forward fill por vaca.

### Justificación
Un diagnóstico positivo de gestación normalmente describe un **estado que persiste** durante un periodo, no solo un evento instantáneo.


In [6]:

if "pregnancy_positive" in df.columns:
    df["pregnancy_positive"] = df["pregnancy_positive"].fillna(0)
    df["is_pregnant"] = (
        df.groupby("cow_id")["pregnancy_positive"]
          .transform(lambda s: s.ffill())
          .fillna(0)
    )
else:
    df["is_pregnant"] = 0

display(df[[c for c in ["cow_id", "date", "pregnancy_positive", "is_pregnant"] if c in df.columns]].head(20))


,cow_id,date,is_pregnant
0,1204,2022-06-05,0
1,1204,2022-06-08,0
2,1204,2022-07-01,0
3,1204,2022-08-15,0
4,1204,2022-09-27,0
5,1204,2023-05-17,0
6,1204,2023-05-23,0
7,1204,2023-06-29,0
8,1204,2023-07-12,0
9,1204,2023-07-13,0



## 7. Variables `days_since_*`

### Decisión
Se generan variables de memoria temporal para eventos clínicos y reproductivos:
- mastitis
- inseminación
- parto
- aborto

### Justificación
Para producción y salud animal, el tiempo desde el último evento suele ser más informativo que el evento puntual.


In [7]:

key_events = [
    "mastitis_flag",
    "insemination_flag",
    "calving_flag",
    "abortion_flag",
]

for col in key_events:
    if col in df.columns:
        df[f"days_since_{col}"] = (
            df.groupby("cow_id", group_keys=False)
              .apply(lambda g: days_since_event_from_date(g, col))
        )

display(
    df[[c for c in ["cow_id", "date"] + [f"days_since_{c}" for c in key_events if f"days_since_{c}" in df.columns]]].head(20)
)


,cow_id,date,days_since_mastitis_flag,days_since_insemination_flag,days_since_calving_flag,days_since_abortion_flag
0,1204,2022-06-05,NaN,NaN,NaN,NaN
1,1204,2022-06-08,NaN,NaN,NaN,NaN
2,1204,2022-07-01,NaN,NaN,NaN,NaN
3,1204,2022-08-15,NaN,NaN,NaN,NaN
4,1204,2022-09-27,NaN,NaN,NaN,NaN
5,1204,2023-05-17,NaN,NaN,NaN,NaN
6,1204,2023-05-23,NaN,NaN,NaN,NaN
7,1204,2023-06-29,NaN,NaN,NaN,NaN
8,1204,2023-07-12,NaN,NaN,NaN,NaN
9,1204,2023-07-13,NaN,NaN,NaN,NaN



## 8. Lags y tendencia reciente de producción

### Decisión
Se agregan:
- `produccion_lag_1`
- `produccion_roll_mean_3`

### Justificación
La producción presente depende fuertemente del historial inmediato de producción.  
Esto ayuda a capturar inercia y tendencia reciente sin inflar demasiado el número de variables.


In [8]:

if "produccion_kg" in df.columns:
    df["produccion_lag_1"] = df.groupby("cow_id")["produccion_kg"].shift(1)

    df["produccion_roll_mean_3"] = (
        df.groupby("cow_id")["produccion_kg"]
          .transform(lambda s: s.shift(1).rolling(window=3, min_periods=1).mean())
    )

display(df[[c for c in ["cow_id", "date", "produccion_kg", "produccion_lag_1", "produccion_roll_mean_3"] if c in df.columns]].head(20))


,cow_id,date,produccion_kg,produccion_lag_1,produccion_roll_mean_3
0,1204,2022-06-05,NaN,NaN,NaN
1,1204,2022-06-08,NaN,NaN,NaN
2,1204,2022-07-01,NaN,NaN,NaN
3,1204,2022-08-15,NaN,NaN,NaN
4,1204,2022-09-27,NaN,NaN,NaN
5,1204,2023-05-17,NaN,NaN,NaN
6,1204,2023-05-23,NaN,NaN,NaN
7,1204,2023-06-29,NaN,NaN,NaN
8,1204,2023-07-12,NaN,NaN,NaN
9,1204,2023-07-13,NaN,NaN,NaN



## 9. Ratio de alimentación

### Decisión
Se crea `ratio_consumo_oferta` si existen los campos necesarios.

### Justificación
Los ratios suelen ser más estables y comparables que las magnitudes absolutas, especialmente cuando el tamaño del lote cambia.


In [9]:

if "consumo_total" in df.columns and "kg_totales_total" in df.columns:
    df["ratio_consumo_oferta"] = df["consumo_total"] / df["kg_totales_total"].replace(0, np.nan)

display(df[[c for c in ["consumo_total", "kg_totales_total", "ratio_consumo_oferta"] if c in df.columns]].head())


,ratio_consumo_oferta
0,NaN
1,NaN
2,NaN
3,NaN
4,NaN



## 10. THI (Temperature Humidity Index)

### Decisión
Se resume el ambiente con `THI`.

### Justificación
THI es una métrica estándar en ganado lechero para representar estrés térmico combinando temperatura y humedad.

Después de construirlo, conviene eliminar temperatura y humedad crudas si ya fueron incorporadas en etapas previas.


In [10]:

if "temperature_2m_mean" in df.columns and "relative_humidity_2m_mean" in df.columns:
    T = df["temperature_2m_mean"]
    RH = df["relative_humidity_2m_mean"]

    df["THI"] = (
        (1.8 * T + 32)
        - (0.55 - 0.0055 * RH) * (1.8 * T - 26)
    )

df["heat_stress_flag"] = (df["THI"] > 72).astype(int) if "THI" in df.columns else 0

display(df[[c for c in ["temperature_2m_mean", "relative_humidity_2m_mean", "THI", "heat_stress_flag"] if c in df.columns]].head())


,THI,heat_stress_flag
0,67.799947,0
1,66.428090,0
2,66.204862,0
3,62.226701,0
4,60.098584,0



## 11. Variables temporales cíclicas

### Decisión
Se generan:
- `dow_sin`, `dow_cos`
- `doy_sin`, `doy_cos`

### Justificación
Los ciclos semanales y anuales no deben representarse como enteros ordinarios, porque eso introduce discontinuidades artificiales.


In [11]:

df["day_of_week"] = df["date"].dt.dayofweek
df["day_of_year"] = df["date"].dt.dayofyear

df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

df["doy_sin"] = np.sin(2 * np.pi * (df["day_of_year"] - 1) / 365)
df["doy_cos"] = np.cos(2 * np.pi * (df["day_of_year"] - 1) / 365)

display(df[["date", "day_of_week", "day_of_year", "dow_sin", "dow_cos", "doy_sin", "doy_cos"]].head())


,date,day_of_week,day_of_year,dow_sin,dow_cos,doy_sin,doy_cos
0,2022-06-05,6,156,-0.781831,0.623490,0.455907,-0.890028
1,2022-06-08,2,159,0.974928,-0.222521,0.409356,-0.912375
2,2022-07-01,4,182,-0.433884,-0.900969,0.025818,-0.999667
3,2022-08-15,0,227,0.000000,1.000000,-0.680773,-0.732494
4,2022-09-27,1,270,0.781831,0.623490,-0.996659,-0.081676



## 12. Alimentación compactada: macros

### Decisión
Se crean `carbs_ratio`, `protein_ratio` y `fiber_ratio` a partir de ingredientes en materia seca relativa.

### Justificación
Las columnas por ingrediente individual pueden ser útiles, pero también generan mucha colinealidad.
Estas variables macro permiten capturar estructura nutricional de forma más compacta.


In [12]:

diet_kg_seca_cols = [c for c in df.columns if c.startswith("diet_kg_seca_") and not c.endswith("_ratio")]

for col in diet_kg_seca_cols:
    if "diet_dm_total" in df.columns:
        df[col + "_ratio"] = df[col] / df["diet_dm_total"].replace(0, np.nan)

def sum_existing(columns):
    existing = [c for c in columns if c in df.columns]
    if not existing:
        return pd.Series(np.nan, index=df.index)
    return df[existing].sum(axis=1)

df["carbs_ratio"] = sum_existing([
    "diet_kg_seca_maiz_molido_ratio",
    "diet_kg_seca_melaza50pct_agua_ratio",
    "diet_kg_seca_oro_milk_ratio",
    "diet_kg_seca_oro_balance_ratio",
])

df["protein_ratio"] = sum_existing([
    "diet_kg_seca_pasta_se_soya_ratio",
])

df["fiber_ratio"] = sum_existing([
    "diet_kg_seca_silo_de_maiz_ratio",
    "diet_kg_seca_silo_de_avena_ratio",
    "diet_kg_seca_triticale_ratio",
    "diet_kg_seca_pata_de_cebada_ratio",
])

display(df[[c for c in ["carbs_ratio", "protein_ratio", "fiber_ratio"] if c in df.columns]].head())


,carbs_ratio,protein_ratio,fiber_ratio
0,NaN,NaN,NaN
1,NaN,NaN,NaN
2,NaN,NaN,NaN
3,NaN,NaN,NaN
4,NaN,NaN,NaN



## 13. Limpieza de columnas redundantes o poco útiles

### Decisión
Se eliminan columnas que:
- son texto o metadata no apta para modelado
- duplican información ya representada mejor
- introducen demasiada granularidad con poco beneficio
- son redundantes con una representación más estable

### Resultado buscado
Un dataset interpretable, relativamente compacto y con menor colinealidad.


In [13]:

drop_cols = []

drop_cols += [
    "event_types",
    "raw_event_text",
    "diagnosis_text",
    "medication_text",
    "treatment_text",
    "diagnosis_categories",
    "treatment_categories",
    "medication_categories",
    "source_file",
]

drop_cols += [c for c in df.columns if c.startswith("diag_")]
drop_cols += [c for c in df.columns if c.startswith("trt_")]
drop_cols += [c for c in df.columns if c.startswith("med_")]

drop_cols += [c for c in df.columns if c.startswith("diet_kg_seca_")]
drop_cols += [c for c in df.columns if "_ratio_ratio" in c]
drop_cols += [c for c in df.columns if c.startswith("diet_kg_humeda_")]
drop_cols += [c for c in df.columns if c.startswith("diet_pct_ms_")]
drop_cols += [c for c in df.columns if c.startswith("diet_kg_am_pm_")]
drop_cols += [c for c in df.columns if c.startswith("usa_")]

drop_cols += [
    "temperature_2m_mean",
    "temperature_2m_min",
    "temperature_2m_max",
    "relative_humidity_2m_mean",
    "pressure_msl_mean",
    "precipitation_sum",
    "wind_speed_10m_mean",
]

drop_cols += [
    "day_of_week",
    "day_of_year",
    "month",
    "day_of_month",
    "days_in_month",
    "month_sin",
    "month_cos",
    "dom_sin",
    "dom_cos",
]

drop_cols += [
    "pregnancy_positive",
    "pregnancy_negative",
    "has_event",
    "event_count",
    "group_id",
    "n_corrales",
    "destino_leche",
    "THI_category",
    "produccion_lag_3",
    "diet_period",
    "kg_am_total",
    "kg_pm_total",
    "kg_totales_total",
    "consumo_total",
    "consumo_prom_corral",
    "kg_am_prom_corral",
    "kg_pm_prom_corral",
    "kg_totales_prom_corral",
    "sobrante_total",
    "rechazo_total",
    "diet_wet_total",
]

drop_cols = sorted(set([c for c in drop_cols if c in df.columns]))
df = df.drop(columns=drop_cols)

print("Columnas eliminadas:", len(drop_cols))
print("Shape después de limpieza:", df.shape)
display(pd.Series(df.columns).to_frame("column").head(100))


Columnas eliminadas: 2
Shape después de limpieza: (18345, 52)


,column
0,cow_id
1,date
2,ordenos_dia
3,produccion_kg
4,duracion_total_min
5,di_mean
6,dd_mean
7,td_mean
8,ti_mean
9,ubre



## 14. Selección final de columnas núcleo

### Decisión
Se define explícitamente un conjunto final de variables.

### Justificación
En lugar de dejar que el notebook termine con lo que “quedó por accidente”, se declara una salida final clara y estable.


In [14]:

final_cols = [
    "cow_id",
    "date",
    "ordenos_dia",
    "produccion_kg",
    "duracion_total_min",
    "di_mean",
    "dd_mean",
    "td_mean",
    "ti_mean",
    "ubre",
    "ms",
    "rumia_min",
    "days_in_milk",
    "last_calving",
    "body_condition_score",
    "weight_kg",
    "liters_at_dryoff",
    "dcc_at_dryoff",
    "del_at_dryoff",
    "gestant_insem_number",
    "mastitis_flag",
    "digestive_flag",
    "vaccine_flag",
    "reproductive_flag",
    "abortion_flag",
    "antibiotic_flag",
    "hormone_flag",
    "dryoff_related_flag",
    "fresh_cow_flag",
    "rumen_bolus_flag",
    "heat_flag",
    "insemination_flag",
    "calving_flag",
    "dryoff_flag",
    "ratio_consumo_oferta_global",
    "ratio_sobrante_oferta_global",
    "ratio_rechazo_consumo_global",
    "consumo_kg_vaca",
    "am_pm_fraction",
    "n_ingredientes",
    "diet_dm_total",
    "days_since_mastitis_flag",
    "days_since_insemination_flag",
    "days_since_calving_flag",
    "days_since_abortion_flag",
    "produccion_lag_1",
    "produccion_roll_mean_3",
    "ratio_consumo_oferta",
    "THI",
    "heat_stress_flag",
    "dow_sin",
    "dow_cos",
    "doy_sin",
    "doy_cos",
    "carbs_ratio",
    "protein_ratio",
    "fiber_ratio",
    "is_pregnant",
]

final_cols = [c for c in final_cols if c in df.columns]
df_final = df[final_cols].copy()

print("Shape final:", df_final.shape)
display(pd.DataFrame({"column": df_final.columns, "dtype": [str(df_final[c].dtype) for c in df_final.columns]}))


Shape final: (18345, 52)


,column,dtype
0,cow_id,Int64
1,date,datetime64[us]
2,ordenos_dia,float64
3,produccion_kg,float64
4,duracion_total_min,float64
5,di_mean,float64
6,dd_mean,float64
7,td_mean,float64
8,ti_mean,float64
9,ubre,float64



## 15. Revisión rápida de faltantes

### Decisión
No se imputan aquí todas las columnas automáticamente.

### Justificación
Algunas variables como `weight_kg` o `body_condition_score` son mediciones eventuales; su imputación depende de la estrategia del modelo.
Este notebook se enfoca en construir la representación, no en fijar una única política de imputación final.


In [15]:

missing_report = (
    pd.DataFrame({
        "column": df_final.columns,
        "nan_count": df_final.isna().sum().values,
        "nan_pct": (df_final.isna().mean() * 100).round(2).values,
        "dtype": [str(df_final[c].dtype) for c in df_final.columns],
    })
    .sort_values(["nan_pct", "nan_count"], ascending=[False, False])
    .reset_index(drop=True)
)

display(missing_report)


,column,nan_count,nan_pct,dtype
0,carbs_ratio,18345,100.00,float64
1,protein_ratio,18345,100.00,float64
2,fiber_ratio,18345,100.00,float64
3,days_since_abortion_flag,16011,87.28,float64
4,days_since_mastitis_flag,12879,70.20,float64
5,rumia_min,12595,68.66,float64
6,days_in_milk,12595,68.66,float64
7,last_calving,12595,68.66,datetime64[us]
8,ti_mean,9206,50.18,float64
9,dd_mean,8758,47.74,float64


In [16]:
drop_cols = [
    "weight_kg",
    "del_at_dryoff",
    "body_condition_score",
    "gestant_insem_number",
    "liters_at_dryoff",
    "dcc_at_dryoff",
]
df_final.drop(columns=drop_cols, inplace=True, errors="ignore")

In [17]:

missing_report = (
    pd.DataFrame({
        "column": df_final.columns,
        "nan_count": df_final.isna().sum().values,
        "nan_pct": (df_final.isna().mean() * 100).round(2).values,
        "dtype": [str(df_final[c].dtype) for c in df_final.columns],
    })
    .sort_values(["nan_pct", "nan_count"], ascending=[False, False])
    .reset_index(drop=True)
)

display(missing_report)


,column,nan_count,nan_pct,dtype
0,carbs_ratio,18345,100.00,float64
1,protein_ratio,18345,100.00,float64
2,fiber_ratio,18345,100.00,float64
3,days_since_abortion_flag,16011,87.28,float64
4,days_since_mastitis_flag,12879,70.20,float64
5,rumia_min,12595,68.66,float64
6,days_in_milk,12595,68.66,float64
7,last_calving,12595,68.66,datetime64[us]
8,ti_mean,9206,50.18,float64
9,dd_mean,8758,47.74,float64



## 16. Guardado del dataset final de feature engineering

Se exporta un parquet listo para:
- entrenamiento
- selección adicional de variables
- EDA final de features


In [18]:

OUT_DIR = Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_PATH = OUT_DIR / "training_dataset.parquet"
df_final.to_parquet(OUT_PATH, index=False)

print("Archivo guardado en:", OUT_PATH.resolve())


Archivo guardado en: /Users/josemanuelromoperedo/Desktop/ProyectoGranja/data/processed/training_dataset.parquet



## 17. Conclusión

El resultado final busca un balance entre:
- **interpretabilidad**
- **señal temporal**
- **contexto fisiológico**
- **robustez para ML**

La decisión central de este notebook es priorizar variables que representen:
1. estado actual de la vaca,
2. memoria de eventos importantes,
3. ambiente resumido,
4. dieta compactada,
5. temporalidad cíclica.
